In [59]:
import pandas as pd
import os
import zipfile

Strategy Used
Calculate volatility, momentum and avg_volume.
Use inverse of volatility instead of volatility for simplicity. 
Normalize all the three metrics.
Now to choose the weights of the metrics I made different combinations of weights and chose the one that yielded the best results by assigning at least some weight to each of the three metrics.
Using these weights I calculated the weights of each of the stocks.

In [61]:
#load the csv file
df = pd.read_csv('combined_close_prices.csv')
#display the file
df.head()

,date,ADANIENT,BEL,Berger_Paints,HCL_Technologies_Ltd,HDFC_Bank_Ltd,Infosys_Ltd,ONGC,Shree_Cements_Ltd,TATAMOTORS,TRENT
0,2021-01-01,475.812983,37.217437,621.115723,796.171937,1382.369212,1156.420312,71.110981,23675.243322,183.808138,687.833703
1,2021-01-04,490.775661,39.678101,626.431285,810.540167,1380.449241,1166.624751,71.338543,23763.839646,190.615840,675.986550
2,2021-01-05,490.775687,39.985686,635.263595,817.470878,1362.401479,1178.575980,73.196906,23821.919150,185.944872,676.882611
3,2021-01-06,494.765721,40.600842,647.285166,843.418260,1377.569276,1195.123928,75.017351,23708.712243,193.249472,675.887048
4,2021-01-07,492.670910,41.339039,645.158945,835.050763,1375.169355,1191.446557,74.334678,24329.882676,195.783750,672.850520


In [63]:
#extract the files from the provided zip file.
zip_path = 'stock_data.zip'
extract_path = 'stock_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

files = os.listdir(extract_path)

In [87]:
#since we need to use both close and volume values we will use the OHLCV data
#and not the dataset just generated.
#create empty dictionary to link stock names to their entire data
ohlcv_data = {}

for file in files:
    file_path = os.path.join(extract_path, file)
    df = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
    stock_name = file.split('.')[0]
    #bring in all the indicators of one stock and not just close value
    ohlcv_data[stock_name] = df


import numpy as np
#make another dictionary to link the indicators for each stock to their stock name
metrics = {}

for stock, df in ohlcv_data.items():
    #returns are the percent change in close prices. We exclude any data points that don't have a value
    returns = df['close'].pct_change().dropna()
    
    #volatility is the standard deviation of returns
    volatility = returns.std()
    
    #momentum is the moving average returns
    momentum = df['close'].rolling(window=20).mean().iloc[-1] - df['close'].iloc[-1]

    #average volume is the mean of volume data points. it indicated liquidity
    avg_volume = df['volume'].mean()
    
    metrics[stock] = {
        'volatility': volatility,
        'momentum': momentum,
        'avg_volume': avg_volume
    }
#printing all the data from each ticker
for stock_name, df in ohlcv_data.items():
    print(f"{stock_name}")
    print(df.head())
    print("\n")


ADANIENT
                  open        high         low       close   volume
date                                                               
2021-01-01  489.927765  492.022545  475.812983  475.812983  5035248
2021-01-04  493.269440  501.349289  485.438968  490.775661  4936582
2021-01-05  493.169708  500.152289  488.381665  490.775687  3654033
2021-01-06  489.678406  499.404143  483.044960  494.765721  3295461
2021-01-07  516.810669  522.646148  492.571152  492.670910  9879731


BEL
                 open       high        low      close     volume
date                                                             
2021-01-01  38.878380  39.370511  37.094406  37.217437  201696168
2021-01-04  40.385540  40.539332  39.032178  39.678101  105614565
2021-01-05  40.447060  40.985327  39.678105  39.985686   64361523
2021-01-06  40.954559  41.646619  39.878024  40.600842   74098809
2021-01-07  40.523945  41.523589  40.385535  41.339039   46011027


Berger_Paints
                  open        h

In [79]:
#make a new data frame for the metrics and take its transpose so that the metrics
#are the columns
metrics_df = pd.DataFrame(metrics).T  

metrics_df.head()


,volatility,momentum,avg_volume
ADANIENT,0.038641,5.312982,4.718551e+06
BEL,0.021782,-16.552497,2.462851e+07
Berger_Paints,0.016504,-15.125632,1.054902e+06
HCL_Technologies_Ltd,0.016212,-61.922319,3.797866e+06
HDFC_Bank_Ltd,0.015052,-47.385441,1.047525e+07


In [81]:
#A better stock has lower volatility, higher momentum and higher volume
#only volatility is low so we can take its inverse to make things simple.
#Then a better stock would require higher inverse of volatility

# Inverting the volatility
metrics_df['inv_volatility'] = 1 / metrics_df['volatility']

# Normalize all metrics to [0, 1] as suggested in the group
for col in ['inv_volatility', 'momentum', 'avg_volume']:
    min_val = metrics_df[col].min()
    max_val = metrics_df[col].max()
    metrics_df[col + '_norm'] = (metrics_df[col] - min_val) / (max_val - min_val)
metrics_df.head()

,volatility,momentum,avg_volume,inv_volatility,inv_volatility_norm,momentum_norm,avg_volume_norm
ADANIENT,0.038641,5.312982,4.718551e+06,25.878919,0.000000,1.000000,0.162218
BEL,0.021782,-16.552497,2.462851e+07,45.910109,0.493908,0.955231,0.853748
Berger_Paints,0.016504,-15.125632,1.054902e+06,60.592000,0.855918,0.958152,0.034968
HCL_Technologies_Ltd,0.016212,-61.922319,3.797866e+06,61.683453,0.882830,0.862336,0.130239
HDFC_Bank_Ltd,0.015052,-47.385441,1.047525e+07,66.435457,1.000000,0.892100,0.362164


In [83]:
#To choose the weights of each metric I am going to try multiple weight combinations
#taking normalized data like had been sugegsted earlier
#naming the columns from the data set for simplification
inv_vol = metrics_df['inv_volatility_norm']
mom = metrics_df['momentum_norm']
vol = metrics_df['avg_volume_norm']

#make dictionaries to store the final chosen weight combinations
weight_combinations = []
#best weight combination will be chosen on the basis of scores
scores = {}

#Trying different combinations of weights that sum up to 1
#w1 = inv_volatility weight, w2 = momentum weight, w3 = volume weight
for w1 in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    for w2 in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        w3 = 1 - (w1 + w2)
        if 0 <= w3 <= 1:
            label = f"I_V:{w1}, M:{w2}, Vol:{w3}"
            weight_combinations.append((w1, w2, w3))
            #we give implicitly that the composite scores have to be calculated for each stock
            composite_score = w1 * inv_vol + w2 * mom + w3 * vol
            scores[label] = composite_score

#Now we hae to display the combinations with the highest score
for label in sorted(scores, key=lambda x: -scores[x].sum())[:5]:
    print(f"\nWeighting: {label}")
    print(scores[label].sort_values(ascending=False))



Weighting: I_V:0.2, M:0.8, Vol:0.0
Berger_Paints           0.937705
Infosys_Ltd             0.917194
HDFC_Bank_Ltd           0.913680
ONGC                    0.895245
HCL_Technologies_Ltd    0.866435
BEL                     0.862966
TATAMOTORS              0.814661
ADANIENT                0.800000
TRENT                   0.735414
Shree_Cements_Ltd       0.150837
dtype: float64

Weighting: I_V:0.3, M:0.7, Vol:0.0
Berger_Paints           0.927482
HDFC_Bank_Ltd           0.924470
Infosys_Ltd             0.914275
HCL_Technologies_Ltd    0.868484
ONGC                    0.854906
BEL                     0.816834
TATAMOTORS              0.759998
TRENT                   0.709184
ADANIENT                0.700000
Shree_Cements_Ltd       0.226255
dtype: float64

Weighting: I_V:0.4, M:0.6, Vol:0.0
HDFC_Bank_Ltd           0.935260
Berger_Paints           0.917259
Infosys_Ltd             0.911356
HCL_Technologies_Ltd    0.870534
ONGC                    0.814567
BEL                     0.770701
TATA

In [85]:
#I want to assign weight to all three metrics and only one of the top five
#weight combinations (0.2,0.7,0.1) does that I am going to use it.
#now we'll add the score column to metrics_df
metrics_df['score'] = (
    0.2 * metrics_df['inv_volatility_norm'] +
    0.7 * metrics_df['momentum_norm'] +
    0.1 * metrics_df['avg_volume_norm']
)

#To get final weights we have to normalize the score column
metrics_df['weight'] = metrics_df['score'] / metrics_df['score'].sum()

#save the file and display it to see the assigned weights
metrics_df.to_csv('metrics_df.csv')
metrics_df.head(10)


,volatility,momentum,avg_volume,inv_volatility,inv_volatility_norm,momentum_norm,avg_volume_norm,score,weight
ADANIENT,0.038641,5.312982,4.718551e+06,25.878919,0.000000,1.000000,0.162218,0.716222,0.096609
BEL,0.021782,-16.552497,2.462851e+07,45.910109,0.493908,0.955231,0.853748,0.852818,0.115034
Berger_Paints,0.016504,-15.125632,1.054902e+06,60.592000,0.855918,0.958152,0.034968,0.845387,0.114031
HCL_Technologies_Ltd,0.016212,-61.922319,3.797866e+06,61.683453,0.882830,0.862336,0.130239,0.793225,0.106995
HDFC_Bank_Ltd,0.015052,-47.385441,1.047525e+07,66.435457,1.000000,0.892100,0.362164,0.860687,0.116095
Infosys_Ltd,0.016095,-32.278516,6.575060e+06,62.130113,0.893843,0.923032,0.226700,0.847561,0.114325
ONGC,0.020367,-6.446247,1.963313e+07,49.098831,0.572532,0.975923,0.680244,0.865677,0.116768
Shree_Cements_Ltd,0.017710,-483.088858,4.812411e+04,56.465993,0.754184,0.000000,0.000000,0.150837,0.020346
TATAMOTORS,0.024282,-31.811813,2.883926e+07,41.183134,0.377355,0.923987,1.000000,0.822262,0.110912
TRENT,0.021189,-98.290181,7.210278e+05,47.194529,0.525578,0.787873,0.023372,0.658964,0.088885
